In [62]:
import numpy as np
import pandas as pd
import json
import seaborn as sns
from pandas import json_normalize
from bs4 import BeautifulSoup
import requests
import warnings
import re
warnings.filterwarnings("ignore")

In [27]:
url_1 = "https://www.jobkaka.com/tag/graduate-jobs"
res = requests.get(url_1)
res

<Response [200]>

In [28]:
print(res.text[:1000])

<!DOCTYPE html>
<html lang="en-US">
<head >
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<link rel="preconnect" href="https://securepubads.g.doubleclick.net">
<link rel="dns-prefetch" href="https://securepubads.g.doubleclick.net">
<link rel="preconnect" href="https://tpc.googlesyndication.com">
<link rel="dns-prefetch" href="https://tpc.googlesyndication.com">
<link rel="preload" as="script" href="https://securepubads.g.doubleclick.net/tag/js/gpt.js">
<script async src="https://securepubads.g.doubleclick.net/tag/js/gpt.js"></script>
<script>
  window.googletag = window.googletag || { cmd: [] };

  googletag.cmd.push(function () {
    // Set base targeting
    googletag.pubads().setTargeting('request_uri', window.location.pathname);

    const urlParams = new URLSearchParams(window.location.search);
    const currentUtmCampaign = urlParams.get('utm_campaign') || 'default_campaign';
    const urlUtmSource = urlParams.get('utm_source');



In [29]:
# class = content_link
# name of job = entry-title
data = res.text
soup = BeautifulSoup(data)
p_n = soup.find("a", class_="content_link")


In [30]:
print(p_n)

<a class="content_link" href="https://www.jobkaka.com/sib-junior-officer-post/"><div class="entry-title">South Indian Bank Recruitment 2026 - Junior Officer Jobs</div>
<div class="bfooter">
<div class="entry-job-liner-1">
<span class="entry-job-date">Job Type </span>
<span class="entry-job-date-details">Government</span>
</div>
<div class="entry-job-liner-1">
<span class="entry-job-date">Qualification </span>
<span class="entry-job-date-details">Graduate</span>
</div>
<div class="entry-job-liner-1">
<span class="entry-job-date">Salary </span>
<span class="entry-job-date-details">Rs.756000/-</span>
</div>
<div align="center" class="entry-job-liner-1">
<button style="padding: 10px"><span class="dashicons dashicons-info-outline"></span> View Details</button>
</div>
</div></a>


In [31]:
j_l = sop.find("a",attrs={"class":"content-link"}) # the link is in this href of a tag 
j_n = sop.find("div",attrs={"class":"entry-title"})
s_n = sop.find("span",attrs={"class":"entry-job-date-details"})
print(j_n)
s_n

<div class="entry-title">Defence Institute of High Altitude Research Recruitment 2026 - Junior Research Fellow Jobs</div>


<span class="entry-job-date-details">Government</span>

In [32]:
job_card = soup.find("a", class_="content_link")

In [33]:
for d in details:
    label_tag = d.find("span", class_="entry-job-date")
    value_tag = d.find("span", class_="entry-job-date-details")

    if label_tag and value_tag:
        label = label_tag.text.strip()
        value = value_tag.text.strip()
        print(label, ":", value)

Job Type : Government
Qualification : Graduate
Salary : Rs.37000-37000/-


In [34]:
job_cards = soup.find_all("a", class_="content_link")

In [38]:
# Title
# Job Type
# Qualification
# Salary
# Link

jobs_data = []

for job in job_cards:
    job_dict = {}
    
    title = job.find("div", class_="entry-title").text.strip()
    link = job["href"]
    
    job_dict["title"] = title
    job_dict["link"] = link
    
    details = job.find_all("div", class_="entry-job-liner-1")
    
    for d in details:
        label_tag = d.find("span", class_="entry-job-date")
        value_tag = d.find("span", class_="entry-job-date-details")
        
        if label_tag and value_tag:
            label = label_tag.text.strip()
            value = value_tag.text.strip()
            job_dict[label] = value
    
    jobs_data.append(job_dict)

In [41]:
print(len(jobs_data))
print(jobs_data[0])

24
{'title': 'South Indian Bank Recruitment 2026 - Junior Officer Jobs', 'link': 'https://www.jobkaka.com/sib-junior-officer-post/', 'Job Type': 'Government', 'Qualification': 'Graduate', 'Salary': 'Rs.756000/-'}


In [ ]:
base_url = "https://www.jobkaka.com/tag/graduate-jobs/"

for page in range(1, 44):  # temporary upper bound
    if page == 1:
        url = base_url
        print(url)
        res = requests.get(url)
    else:
        url = base_url + f"page/{page}/"
        print(url)
        res = requests.get(url)
        if res.text == None:
            break

In [46]:
import time

all_jobs = []

for page in range(1, 50):
    if page == 1:
        url = base_url
    else:
        url = base_url + f"page/{page}/"

    # the page we are scraping at the loop
    print("Scraping:", url)
    res = requests.get(url)
    if res.status_code != 200:
        break
    
    soup = BeautifulSoup(res.text, "html.parser")
    job_cards = soup.find_all("a", class_="content_link")
    if len(job_cards) == 0:
        print("No more jobs. Stopping.")
        break
    
    # extracting jobs 
    # ---------------------------------
    for job in job_cards:
        job_dict = {}
        
        title = job.find("div", class_="entry-title").text.strip()
        link = job["href"]
        
        job_dict["title"] = title
        job_dict["link"] = link
        
        details = job.find_all("div", class_="entry-job-liner-1")
        
        for d in details:
            label_tag = d.find("span", class_="entry-job-date")
            value_tag = d.find("span", class_="entry-job-date-details")
            
            if label_tag and value_tag:
                label = label_tag.text.strip()
                value = value_tag.text.strip()
                job_dict[label] = value
        
        all_jobs.append(job_dict)
    
    time.sleep(1)


print(len(all_jobs))

Scraping: https://www.jobkaka.com/tag/graduate-jobs/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/2/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/3/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/4/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/5/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/6/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/7/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/8/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/9/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/10/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/11/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/12/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/13/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/14/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/15/
Scraping: https://www.jobkaka.com/tag/graduate-jobs/page/16/
Scraping: https://www.jobkaka.com/tag/gr

In [49]:
print(all_jobs[0].keys())
for i in range(5):
    print(all_jobs[i].keys())

dict_keys(['title', 'link', 'Job Type', 'Qualification', 'Salary'])
dict_keys(['title', 'link', 'Job Type', 'Qualification', 'Salary'])
dict_keys(['title', 'link', 'Job Type', 'Qualification', 'Salary'])
dict_keys(['title', 'link', 'Job Type', 'Qualification', 'Salary'])
dict_keys(['title', 'link', 'Job Type', 'Qualification', 'Salary'])
dict_keys(['title', 'link', 'Job Type', 'Qualification', 'Salary'])


In [50]:
df = pd.DataFrame(all_jobs)

In [51]:
df.shape

(1028, 5)

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1028 entries, 0 to 1027
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   title          1028 non-null   object
 1   link           1028 non-null   object
 2   Job Type       1028 non-null   object
 3   Qualification  1028 non-null   object
 4   Salary         1028 non-null   object
dtypes: object(5)
memory usage: 40.3+ KB


In [54]:
df.head()

,title,link,Job Type,Qualification,Salary
0,South Indian Bank Recruitment 2026 - Junior Of...,https://www.jobkaka.com/sib-junior-officer-post/,Government,Graduate,Rs.756000/-
1,CSIR-National Aerospace Laboratories Recruitme...,https://www.jobkaka.com/csir-nal-project-assoc...,Government,Graduate,Rs.25000-25000/-
2,Fertilizer Corporation of India Limited (FCIL)...,https://www.jobkaka.com/fcil-young-professiona...,Government,Graduate,Rs.30000-60000/-
3,Arunachal Pradesh Public Service Commission Re...,https://www.jobkaka.com/appsc-assistant-profes...,Government,Master's,Rs.57700-182400/-
4,Hindustan Petroleum Corporation Limited (HPCL)...,https://www.jobkaka.com/hpcl-graduate-apprenti...,Government,Graduate,Rs.25000-25000/-


In [55]:
df.isnull().sum()

title            0
link             0
Job Type         0
Qualification    0
Salary           0
dtype: int64

### Raw Data of Job Details

In [56]:
df.to_csv("raw_job_data.csv", index=False)

### Python Sql Data loading 

In [ ]:
!pip install mysql-connector-python

In [ ]:
import mysql.connector
import pandas as pd

# Load cleaned CSV
df = pd.read_csv("jobs_cleaned.csv")

# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="your_username",
    password="your_password",
    database="your_database"
)

cursor = conn.cursor()

insert_query = """
INSERT INTO jobs_data
(title, link, job_type, qualification, qualification_level, salary_minimum, salary_maximum)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""

for _, row in df.iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()

cursor.close()
conn.close()